# GraphGuard: 02 - Exploratory Data Analysis (EDA)
**Objective**: Analyze transaction amount distributions, diurnal spending curves, merchant category risks, and geographical channels.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_path = os.path.join("..", "data", "processed", "transactions_cleaned.csv")
df = pd.read_csv(cleaned_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Loaded cleaned transactions: {len(df):,} records")

### 1. Transaction Amount Distribution (Log-Normal vs Fraud Spike)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df["is_fraud"] == 0]["amount"], bins=50, kde=True, ax=ax[0], color="#2563EB")
ax[0].set_title("Legitimate Transaction Amount Distribution")
ax[0].set_xlabel("Amount ($)")

sns.histplot(df[df["is_fraud"] == 1]["amount"], bins=50, kde=True, ax=ax[1], color="#DC2626")
ax[1].set_title("Fraudulent Transaction Amount Distribution")
ax[1].set_xlabel("Amount ($)")
plt.tight_layout()
plt.show()

### 2. Diurnal Temporal Spending Patterns

In [ ]:
df["hour"] = df["timestamp"].dt.hour
hourly_fraud = df.groupby(["hour", "is_fraud"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
hourly_fraud[0].plot(ax=ax, label="Legitimate", color="#10B981", lw=2)
(hourly_fraud[1] * 10).plot(ax=ax, label="Fraud (Scaled 10x)", color="#EF4444", lw=2.5, linestyle="--")
ax.set_title("Hourly Distribution: Legitimate vs Fraud Activity")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Transaction Count")
ax.legend()
plt.show()

### 3. Fraud Risk by Merchant Category

In [ ]:
cat_summary = df.groupby("merchant_category").agg(
    total=("transaction_id", "count"),
    fraud=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean")
).sort_values("fraud_rate", ascending=False)
print("Merchant Category Risk Breakdown:")
cat_summary